# OpenRouter Free Models Router

Интерактивный smoke playbook. Конкретная бесплатная модель выбирается OpenRouter, поэтому результат не полностью воспроизводим; availability и rate limits контролируются OpenRouter.

## Input parameters

In [ ]:
arsenal_config_path = "@comp/arsenal_openrouter_free.toml"
arsenal_start_and_stop_at_job_level = False
prompt = "Hello"

## Playbook preparation

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path

while not Path(".zemicomp").is_file():
    if Path.cwd().parent == Path.cwd():
        raise FileNotFoundError("Could not find the ZEMI component root")
    %cd ..
PROJECT_ROOT = Path.cwd()

import zemi
from zemi.arsenal import ArsenalSession
from zemi.arsenal.python import PythonVenv
from zemi.playbook import output_params

PythonVenv.from_config("@comp/00_init.toml").verify()

## Starting Arsenal

In [ ]:
arsenal = ArsenalSession(arsenal_config_path)
zemi.arsenal.begin(
    arsenal,
    stop_before_begin=not arsenal_start_and_stop_at_job_level,
)

## Model request

При первом доступе `OPENROUTER_API_KEY` запрашивается через `getpass` и сохраняется только в `@inst/_secrets/arsenal.env`.

In [ ]:
model = arsenal.endpoints.openrouter.models.openrouter_free
assistant = model.assistants.assistant
model

In [ ]:
client = assistant.clients.openai.client
client

In [ ]:
response = client.chat.completions.create(
    model=model.model,
    messages=[{"role": "user", "content": prompt}],
)

In [ ]:
model_response = response.choices[0].message.content
print(model_response)

## Output parameters

In [ ]:
output_params({"model_response": model_response, "response_length": len(model_response)})

## Stopping Arsenal

In [ ]:
zemi.arsenal.end(
    arsenal,
    stop_after_end=not arsenal_start_and_stop_at_job_level,
)